In [1]:
from nba_api.stats.endpoints import leaguedashlineups
import pandas as pd
import time

seasons = ['2025-26'] # Changed to a list
all_lineups = []

for season in seasons: # Iterate over the list
    for group_size in [2, 3, 4, 5]:
        data = leaguedashlineups.LeagueDashLineups(
            season=season,
            measure_type_detailed_defense='Advanced',
            per_mode_detailed='PerGame',
            group_quantity=group_size,
        ).get_data_frames()[0]

        data['season'] = season
        data['group_size'] = group_size
        all_lineups.append(data)
        time.sleep(1)

df = pd.concat(all_lineups)
df.to_csv('nba_lineups.csv', index=False)


In [2]:
df.shape

(8000, 52)

In [3]:
df.columns

Index(['GROUP_SET', 'GROUP_ID', 'GROUP_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION',
       'GP', 'W', 'L', 'W_PCT', 'MIN', 'E_OFF_RATING', 'OFF_RATING',
       'E_DEF_RATING', 'DEF_RATING', 'E_NET_RATING', 'NET_RATING', 'AST_PCT',
       'AST_TO', 'AST_RATIO', 'OREB_PCT', 'DREB_PCT', 'REB_PCT', 'TM_TOV_PCT',
       'EFG_PCT', 'TS_PCT', 'E_PACE', 'PACE', 'PACE_PER40', 'POSS', 'PIE',
       'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK',
       'OFF_RATING_RANK', 'DEF_RATING_RANK', 'NET_RATING_RANK', 'AST_PCT_RANK',
       'AST_TO_RANK', 'AST_RATIO_RANK', 'OREB_PCT_RANK', 'DREB_PCT_RANK',
       'REB_PCT_RANK', 'TM_TOV_PCT_RANK', 'EFG_PCT_RANK', 'TS_PCT_RANK',
       'PACE_RANK', 'PIE_RANK', 'SUM_TIME_PLAYED', 'season', 'group_size'],
      dtype='str')

## Clean the data

In [4]:
from src.get_data import get_player_stats, compute_expected_metric, get_lineups, parse_player_ids, compute_weighted_ts, compute_expected_net

In [5]:
SEASONS       = ['2025-26']          # add more e.g. ['2022-23', '2023-24', '2024-25', '2025-26']
GROUP_SIZES   = [2, 3, 4, 5]
MIN_THRESHOLDS = {2: 100, 3: 75, 4: 50, 5: 30}   # minimum shared minutes to include a lineup

In [6]:
all_lineups = []
all_players = []

for season in SEASONS:
    print(f"\n{'='*50}")
    print(f"Season: {season}")
    print(f"{'='*50}")

    # 1. Individual player stats
    player_df = get_player_stats(season)
    all_players.append(player_df)
    time.sleep(1)

    # 2. Build lookup dicts: player_id → metric
    player_net_lookup = dict(zip(player_df['PLAYER_ID'], player_df['NET_RATING']))
    player_off_lookup = dict(zip(player_df['PLAYER_ID'], player_df['OFF_RATING']))
    player_def_lookup = dict(zip(player_df['PLAYER_ID'], player_df['DEF_RATING']))
    player_pie_lookup = dict(zip(player_df['PLAYER_ID'], player_df['PIE']))
    player_ts_lookup  = dict(zip(player_df['PLAYER_ID'], player_df['TS_PCT']))
    player_usg_lookup = dict(zip(player_df['PLAYER_ID'], player_df['USG_PCT']))

    # 3. Lineup data for each group size
    for group_size in GROUP_SIZES:
        df = get_lineups(season, group_size)

        # Parse player IDs
        df['player_ids'] = df['GROUP_ID'].apply(parse_player_ids)

        # Apply minimum minutes filter
        min_thresh = MIN_THRESHOLDS[group_size]
        df = df[df['MIN'] >= min_thresh].copy()
        print(f"    {len(df)} lineups after {min_thresh}min filter")

        # Compute expected values from individual stats
        df['expected_net_rating'] = df['player_ids'].apply(
            lambda ids: compute_expected_net(ids, player_net_lookup)
        )
        df['expected_off_rating'] = df['player_ids'].apply(
            lambda ids: compute_expected_metric(ids, player_off_lookup)
        )
        df['expected_def_rating'] = df['player_ids'].apply(
            lambda ids: compute_expected_metric(ids, player_def_lookup)
        )
        df['expected_pie'] = df['player_ids'].apply(
            lambda ids: compute_expected_metric(ids, player_pie_lookup)
        )
        df['expected_ts_usg_weighted'] = df['player_ids'].apply(
            lambda ids: compute_weighted_ts(ids, player_ts_lookup, player_usg_lookup)
        )

        # Compute synergy deltas
        df['synergy_delta']     = df['NET_RATING'] - df['expected_net_rating']
        df['off_synergy_delta'] = df['OFF_RATING'] - df['expected_off_rating']
        df['def_synergy_delta'] = df['DEF_RATING'] - df['expected_def_rating']   # lower is better
        df['pie_synergy_delta'] = df['PIE']        - df['expected_pie']

        # Convert player_ids list → string for CSV export
        df['player_ids_str'] = df['player_ids'].apply(lambda x: ','.join(x))
        df = df.drop(columns=['player_ids'])

        all_lineups.append(df)


Season: 2025-26
  Fetching individual player stats for 2025-26...
  Fetching 2-man lineups for 2025-26...
    2166 lineups after 100min filter
  Fetching 3-man lineups for 2025-26...
    3256 lineups after 75min filter
  Fetching 4-man lineups for 2025-26...
    1861 lineups after 50min filter
  Fetching 5-man lineups for 2025-26...
    397 lineups after 30min filter


In [16]:
player_df.shape

(546, 15)

In [24]:
df_combined = pd.concat(all_lineups[0:4], ignore_index=True)
df_combined.to_csv('lineups_with_synergy.csv', index=False)

### Create clusters

In [1]:
from src.cluster_creation import load_data, cluster_players, plot_clusters, build_archetype_lookup, enrich_lineups, archetype_compatibility_matrix, best_lineups_per_team
import json

In [2]:
players_df, lineups_df = load_data()
players_with_archetypes, scaler, km = cluster_players(players_df)

Loading data...
  546 player-season rows
  7680 lineup rows

  440 players with 15+ games (used for clustering)

── Cluster Centroids (original scale) ──────────────────────────────
         USG_PCT  AST_PCT  OREB_PCT  DREB_PCT  TS_PCT    PIE  REB_PCT
cluster                                                              
0          0.156    0.103     0.029     0.090   0.572  0.070    0.060
1          0.145    0.088     0.120     0.191   0.634  0.109    0.155
2          0.174    0.217     0.027     0.102   0.487  0.072    0.064
3          0.267    0.225     0.066     0.213   0.593  0.151    0.140
4          0.179    0.120     0.053     0.146   0.598  0.101    0.100
5          0.131    0.083     0.063     0.136   0.500  0.061    0.099
6          0.246    0.251     0.025     0.105   0.582  0.120    0.065

── Top players per cluster (by PIE) ───────────────────────────────
  [0] 3-and-D Wing             : Jamaree Bouyea, Mikal Bridges, Zach LaVine, Isaiah Joe, Jerami Grant
  [1] Rim-Runner 

In [42]:
plot_clusters(players_with_archetypes)

  Saved player_archetypes_pca.png


In [ ]:
archetype_lookup = build_archetype_lookup(players_with_archetypes)

In [5]:
lineups_enriched = enrich_lineups(lineups_df, archetype_lookup)
n_matched = lineups_enriched['archetype_fingerprint'].notna().sum()

In [7]:
compatibility = archetype_compatibility_matrix(lineups_enriched)


── Archetype Compatibility Matrix (avg synergy_delta for 2-man lineups) ──
                      3-and-D Wing  Rim-Runner  Secondary Creator  Star Engine  Versatile Big  Glue / Role Player  Primary Ball-Handler
3-and-D Wing                 -1.74        1.07              -0.51         0.68          -0.24               -0.17                 -0.02
Rim-Runner                    1.07         NaN              -1.17        -0.08          -0.46                1.37                  0.51
Secondary Creator            -0.51       -1.17              -2.22        -0.57          -1.47                0.88                 -0.43
Star Engine                   0.68       -0.08              -0.57          NaN           2.49               -0.72                  1.24
Versatile Big                -0.24       -0.46              -1.47         2.49           0.11               -1.92                  0.04
Glue / Role Player           -0.17        1.37               0.88        -0.72          -1.92               

In [8]:
best_lineups_per_team(lineups_enriched, group_size=5)
# best_lineups_per_team(lineups_enriched, group_size=2)


── Top 3 lineups (group_size=5) per team by synergy_delta ──
  Saved best_lineups_per_team.csv (0 rows)


""


In [39]:
ARCHETYPE_LABELS = {
    0: "Primary Ball-Handler",    # high USG, high AST
    1: "3-and-D Wing",            # low USG, high TS, moderate DEF value
    2: "Stretch Big",             # high TS, moderate REB, perimeter capable
    3: "Rim-Runner",              # high OREB/DREB, low USG, paint-only
    4: "Secondary Creator",       # moderate USG, moderate AST, shooting guard type
    5: "Two-Way Wing",            # balanced across everything, high PIE
    6: "Glue Guy",                # low across the board but high team impact
}


In [40]:
fives = lineups_enriched[
        (lineups_enriched['group_size'] == 5) &
        (lineups_enriched['archetype_fingerprint_str'].notna())]

combo_summary = (
    fives.groupby('archetype_fingerprint_str')['synergy_delta']
            .agg(mean='mean', std='std', count='count')
            .query('count >= 3')
            .sort_values('mean', ascending=False)
            .head(10))

# Decode fingerprint to labels for readability
def decode(fp_str):
    try:
        ids = json.loads(fp_str.replace('(', '[').replace(')', ']'))
        return ' + '.join(ARCHETYPE_LABELS.get(i, str(i)) for i in ids)
    except Exception:
        return fp_str

combo_summary['archetype_names'] = combo_summary.index.map(decode)
print(combo_summary[['archetype_names', 'mean', 'std', 'count']].round(3).to_string())

                                                                                                 archetype_names    mean     std  count
archetype_fingerprint_str                                                                                                              
(0, 0, 1, 2, 3)            Primary Ball-Handler + Primary Ball-Handler + 3-and-D Wing + Stretch Big + Rim-Runner  28.713  26.777      3
(0, 1, 2, 3, 6)                        Primary Ball-Handler + 3-and-D Wing + Stretch Big + Rim-Runner + Glue Guy  25.600   6.971      3
(2, 2, 3, 4, 6)                            Stretch Big + Stretch Big + Rim-Runner + Secondary Creator + Glue Guy  24.307   7.025      3
(1, 3, 5, 6, 6)                                   3-and-D Wing + Rim-Runner + Two-Way Wing + Glue Guy + Glue Guy  21.180  16.789      3
(0, 0, 4, 6, 6)            Primary Ball-Handler + Primary Ball-Handler + Secondary Creator + Glue Guy + Glue Guy  21.067  27.755      3
(0, 1, 5, 6, 6)                         Primary 